# Lesson 88 · TF-IDF retrieval from scratch — teaching a machine to find a needle in a haystack (pure NumPy)

**Learning goals**
- Understand how the TF-IDF sparse representation works
- Implement build_tfidf() and cosine_retrieve() from scratch
- Check them against aurora.llm.retrieve
- Use them to search a small DSP knowledge base

> Classic IR: term frequency × inverse document frequency; compare with embedding retrieval (dense vs sparse).

← **Previous lesson** · [L87 · Quantization and local inference](L87_local_inference.ipynb)

> Last lesson covered **quantization and local inference**: how INT8 quantization works, and lightweight inference with Hugging Face.
> This lesson explores **TF-IDF retrieval from scratch**.

## How does a search engine know you're after "Fourier transform" and not "Eiffel Tower"?

The simplest version: **term frequency × rarity**.

**TF (term frequency)**: if "Fourier" shows up 5 times in an article, the article is probably really about the Fourier transform.

**IDF (inverse document frequency)**:
- "the" appears in every article → very low IDF → very low weight
- "STFT" appears only in a handful of technical articles → high IDF → high weight

**Put together**: `TF-IDF(term, document) = TF × IDF`

**The intuition as formulas**:
```
TF(t, d)  = count(t, d) / |d|               → relative frequency of term t in document d
IDF(t)    = log((N+1)/(df(t)+1)) + 1         → N = number of documents, df = documents containing the term
TF-IDF    = TF × IDF                         → frequent + rare → high weight
```

**The retrieval flow**:
```
query "Fourier transform"
  1. tokenize → ["fourier", "transform"]
  2. build the query vector q (TF only, no IDF)
  3. compute the cosine similarity between q and every document vector
  4. return the top-k most similar documents
```

**Why not just use sentence embeddings?** TF-IDF needs no pretrained weights at all — you can derive the whole thing from the formula — it's good enough for keyword matching, and inside a RAG pipeline it is often easier to explain than vector search.

In this lesson you implement `tokenize` → `build_tfidf` → `cosine_retrieve` and check them against `aurora.llm.retrieve`.

### 🤔 What is that log doing in the IDF?

Why does the formula `IDF = log((N+1)/(df+1)) + 1` wrap everything in a **log**? (Throughout this lesson, log always means the **natural log ln** — in code, `np.log`.) Because the value of rarity has diminishing returns: a word going from "appears in 100 documents" to "appears in 10" really does become more discriminating, but going from 10 down to 1 doesn't deserve a hundredfold reward. The log squashes the ratio `N/df`, so the gap between common and rare words lands in a sensible range instead of letting extremely rare words' weights explode. The `+1` smoothing term guarantees every word keeps at least some base weight (so an IDF of 0 never wipes out a whole dimension).

In [ ]:
import numpy as np
import re
from collections import Counter

In [ ]:
# A small DSP / audio-AI knowledge base (10 documents)
DOCS = [
    'The Fourier transform decomposes a signal into its frequency components',
    'FFT stands for Fast Fourier Transform and runs in O N log N time',
    'The STFT applies a windowed Fourier transform to produce a spectrogram',
    'Mel filterbanks simulate human auditory perception of frequency',
    'MFCC Mel Frequency Cepstral Coefficients are computed via DCT after mel filterbank',
    'The Nyquist theorem states the sampling rate must be at least twice the maximum frequency',
    'Window functions like Hann and Hamming reduce spectral leakage in FFT',
    'CTC Connectionist Temporal Classification allows sequence labeling without alignment',
    'Whisper is a Transformer based speech recognition model from OpenAI',
    'Beat tracking estimates the tempo of music using onset detection and autocorrelation',
]
print(f'Knowledge base: {len(DOCS)} documents')

## ✏️ Task 1: implement `tokenize`

**Signature**: `tokenize(text: str) -> list[str]`

**Two-step route**:

| Step | What to do | Code hint |
|---|---|---|
| 1 | Pull out ASCII words | `re.findall(r'[a-z]+', ...)` |
| 2 | Lowercase | call `.lower()` before findall, or use the `re.I` flag in the pattern |

**Acceptance criteria**:
- `tokenize('Hello World!')` == `['hello', 'world']`
- `tokenize('FFT O(N log N)')` == `['fft', 'o', 'n', 'log', 'n']` (parentheses and capitals both handled correctly)

In [ ]:
def tokenize(text: str) -> list:
    """A simple tokenizer: extract lowercase ASCII words."""
    # ✏️ TODO: extract words with a regex, lowercase them
    raise NotImplementedError("TODO")

assert tokenize('Hello World!') == ['hello', 'world']
assert tokenize('FFT O(N log N)') == ['fft', 'o', 'n', 'log', 'n']
print('✅ tokenize check passed')

## ✏️ Task 2: implement `build_tfidf`

**Signature**: `build_tfidf(docs: list) -> (matrix, vocab)`
- `matrix` shape: `(n_docs, vocab_size)`, values are TF-IDF weights
- `vocab`: the vocabulary as a list (sorted)

**Three-step route**:

| Step | What to do | Formula |
|---|---|---|
| 1 | Compute the TF matrix | `TF[d, t] = count(t, d) / len(tokens_d)` |
| 2 | Compute the IDF vector | `IDF[t] = log((N+1) / (df(t)+1)) + 1` (log is the natural log ln, i.e. `np.log`); `df(t)` = number of documents containing term t |
| 3 | Return TF × IDF | `matrix = TF * IDF` (broadcasting: `TF (n_docs, V)` × `IDF (V,)`) |

**Acceptance criteria**:
- `matrix.shape == (len(docs), len(vocab))`
- Frequent-and-rare terms get the highest weights (print the top terms to check)
- Numerically identical to `aurora.llm.build_tfidf` (atol=1e-5)

In [ ]:
def build_tfidf(docs: list) -> tuple:
    """Build the TF-IDF matrix. Returns (matrix, vocab)."""
    tokenized = [tokenize(d) for d in docs]
    all_terms = sorted({t for tokens in tokenized for t in tokens})
    word_idx = {w: i for i, w in enumerate(all_terms)}

    # ✏️ TODO:
    # (1) compute the TF matrix (n_docs, vocab_size): TF(t,d) = count(t,d) / |d|
    # (2) compute the IDF vector (vocab_size,): IDF(t) = log((N+1)/(df(t)+1)) + 1
    #     df(t) = number of documents containing term t, N = total number of documents
    # (3) return (TF × IDF, all_terms) — the TF-IDF matrix and the vocabulary list
    raise NotImplementedError("please implement build_tfidf")

try:
    matrix, vocab = build_tfidf(DOCS)
    print(f'TF-IDF matrix shape: {matrix.shape}  (n_docs={len(DOCS)}, vocab={len(vocab)})')
    print(f'First 10 vocabulary terms: {vocab[:10]}')
except (NotImplementedError, TypeError):
    matrix = None
    vocab = []
    print("⚠️  build_tfidf isn't implemented yet; matrix/vocab set to placeholders. Re-run this cell after finishing the TODO.")


## ✏️ Task 3: implement cosine_retrieve

**The cosine similarity formula**
$$\cos(q,\,d) = \frac{q \cdot d}{\|q\|\,\|d\|}$$

- $q$ — the query vector (TF only, no IDF). Why no IDF on the query side? Because the document side's TF-IDF has already turned down the contribution of common words — in the dot product each term scores ∝ TF_q × TF_d × IDF, and multiplying the query by IDF again would square the weight of rare terms (IDF²) and throw things off balance. This is also common practice in retrieval systems.
- $d$ — a document row vector (TF-IDF)
- When the denominator is zero (an empty query or an all-zero document), the score is 0

Implementation steps:
1. Build $q$ from the query terms' TF (terms not in the vocabulary are simply ignored)
2. Compute the dot products $\text{scores}[i] = \text{tfidf\_matrix}[i] \cdot q$
3. Divide by the respective L2 norms (zero vectors score 0)
4. Take the top-k sorted by descending score and return `[(doc, score), ...]`


In [ ]:
def cosine_retrieve(query: str, tfidf_matrix, vocab: list, docs: list, top_k=3) -> list:
    """Cosine-similarity retrieval; returns a list of (doc, score)."""
    word_idx = {w: i for i, w in enumerate(vocab)}
    tokens = tokenize(query)
    counts = Counter(tokens)
    total = max(len(tokens), 1)

    # ✏️ TODO:
    # (1) build the query vector q_vec (vocab_size,): q_vec[word_idx[t]] = counts[t] / total
    #     ignore terms that aren't in word_idx
    # (2) compute the dot products: scores = tfidf_matrix @ q_vec  → shape (n_docs,)
    # (3) cosine-normalize: score_i /= (‖tfidf_matrix[i]‖ × ‖q_vec‖)
    #     if ‖q_vec‖ == 0 or ‖doc_vec‖ == 0, set that document's score to 0
    # (4) take top_k: sort by descending score, return [(docs[i], float(scores[i])), ...]
    raise NotImplementedError("please implement cosine_retrieve")

# Test (finish build_tfidf and cosine_retrieve first)
if matrix is not None:
    try:
        print('\nQuery: "Fourier transform frequency"')
        for doc, score in cosine_retrieve('Fourier transform frequency', matrix, vocab, DOCS):
            print(f'  [{score:.3f}] {doc[:60]}')
        print('\nQuery: "speech recognition neural network"')
        for doc, score in cosine_retrieve('speech recognition neural network', matrix, vocab, DOCS):
            print(f'  [{score:.3f}] {doc[:60]}')
    except (NotImplementedError, TypeError):
        print("⚠️  cosine_retrieve isn't implemented yet; re-run this cell after finishing the TODO.")
else:
    print("⚠️  matrix is None — finish build_tfidf first.")


## 🔗 Aurora connection

The two functions in this lesson correspond to `src/aurora/llm/retrieve.py`:

| This lesson | Production code |
|---|---|
| `build_tfidf(docs)` | `aurora.llm.build_tfidf` |
| `cosine_retrieve(query, ...)` | `aurora.llm.cosine_retrieve` |

Next lesson (L89) does `from aurora.llm import ...` directly and plugs this retriever into a complete RAG pipeline.


In [ ]:
# Check against aurora.llm (finish both TODOs first)
from aurora.llm import build_tfidf as ref_build, cosine_retrieve as ref_retrieve

ref_mat, ref_vocab = ref_build(DOCS)
print(f'aurora.llm matrix shape: {ref_mat.shape}  ← should match your own implementation')

if matrix is not None:
    # 1. Compare the matrix values
    assert np.allclose(matrix, ref_mat, atol=1e-5), (
        f"❌ The matrix doesn't match the reference implementation!\n"
        f"  yours     shape={matrix.shape}, max={matrix.max():.4f}\n"
        f"  reference shape={ref_mat.shape}, max={ref_mat.max():.4f}\n"
        "Check that your TF and IDF formulas match the ones in Task 2."
    )
    print("✅ build_tfidf matrix matches the reference implementation")

    # 2. Compare the retrieval order
    try:
        my_results = cosine_retrieve('Fourier transform frequency', matrix, vocab, DOCS)
        ref_results = ref_retrieve('Fourier transform frequency', ref_mat, ref_vocab, DOCS)
        my_docs = [d for d, _ in my_results]
        ref_docs = [d for d, _ in ref_results]
        assert my_docs == ref_docs, (
            f"❌ The retrieval order doesn't match the reference implementation!\n"
            f"  yours:     {my_docs}\n"
            f"  reference: {ref_docs}"
        )
        print("✅ cosine_retrieve results are in the same order as the reference implementation")
        print("✅ TF-IDF retrieval check passed")
    except (NotImplementedError, TypeError):
        print("⚠️  cosine_retrieve isn't implemented yet; skipping the retrieval-order comparison.")
else:
    # Just show the reference results for comparison
    ref_results = ref_retrieve('Fourier transform frequency', ref_mat, ref_vocab, DOCS)
    print("aurora.llm retrieval results (reference):", [s for _, s in ref_results])
    print("⚠️  build_tfidf isn't implemented yet; skipping the comparison with your implementation.")


## ✏️ Closed-book derivation check cell — TF-IDF by hand

**Rule: close every cell above and work the following out from memory.**

**Problem**: there are 3 documents and the vocabulary has only 4 words: `{cat, dog, the, run}`

| Document | Content |
|---|---|
| D0 | "the cat run" |
| D1 | "the dog run run" |
| D2 | "cat cat dog" |

1. Compute each word's `df` (the number of documents containing it)
2. Compute the `IDF` (with the formula `log((3+1)/(df+1)) + 1`, log being the natural log ln)
3. Compute D1's TF vector
4. Compute D1's TF-IDF vector
5. For the query "cat run", which document has the highest cosine similarity? (by hand or by estimate)

(Write your derivation here...)

In [ ]:
# ✏️ Self-review for this lesson
l88_review = {
    "tfidf_intuition":           None,  # understand TF = term frequency, IDF = rarity, combined = frequent + rare → high weight? True/False
    "tokenize_impl":             None,  # tokenize() is correct and both asserts pass? True/False
    "build_tfidf_impl":          None,  # build_tfidf() is correct and matches aurora.llm? True/False
    "cosine_retrieve_impl":      None,  # cosine_retrieve() is correct and the retrieval order lines up? True/False
    "whiteboard_passed":         None,  # cleared the whiteboard challenge of working out TF-IDF by hand? True/False
}

unfilled = [k for k, v in l88_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l88_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L88 all clear! On to L89: a complete RAG pipeline')

In [ ]:
# Check: TF-IDF values worked out by hand (the 3-document × 4-word mini example)
import numpy as np

MINI_DOCS = ["the cat run", "the dog run run", "cat cat dog"]
MINI_VOCAB = ["cat", "dog", "run", "the"]  # sorted

# TF by hand
tf_manual = np.array([
    [1/3, 0,   1/3, 1/3],   # D0: cat=1/3, dog=0, run=1/3, the=1/3
    [0,   1/4, 2/4, 1/4],   # D1: cat=0, dog=1/4, run=2/4, the=1/4
    [2/3, 1/3, 0,   0  ],   # D2: cat=2/3, dog=1/3, run=0, the=0
])

# df per word: cat in D0,D2 → df=2; dog in D1,D2 → df=2; run in D0,D1 → df=2; the in D0,D1 → df=2
df = np.array([2, 2, 2, 2])
N = 3
idf_manual = np.log((N + 1) / (df + 1)) + 1   # smoothed

tfidf_manual = tf_manual * idf_manual

# D0 TF-IDF
assert tfidf_manual.shape == (3, 4), f"shape should be (3,4), got {tfidf_manual.shape}"
assert tfidf_manual[0, 0] > 0, "D0's weight for cat should be > 0"

# Self-consistency: weight = TF × IDF; common words get a lower IDF
print("IDF per word (cat/dog/run/the):", [f"{v:.4f}" for v in idf_manual])
print("D1 TF-IDF:", [f"{v:.4f}" for v in tfidf_manual[1]])

# Check build_tfidf's result (if implemented)
try:
    matrix, vocab = build_tfidf(MINI_DOCS)
    assert np.allclose(matrix, tfidf_manual, atol=1e-9), \
        f"Doesn't match the hand calculation!\nyours:\n{matrix}\nby hand:\n{tfidf_manual}"
    print("✅ build_tfidf matches the hand calculation exactly")
except (NotImplementedError, TypeError):
    print("⚠️ build_tfidf isn't implemented yet; the hand calculation check is done")
except Exception as e:
    print(f"⚠️ Check raised an exception (build_tfidf not finished yet): {e}")

## Wrap-up

| Concept | What to remember |
|---|---|
| TF | the term's relative frequency in a document |
| IDF | the more documents contain the term, the lower its weight (common words weigh little) |
| Cosine similarity | same direction → similar, regardless of vector length |
| Strengths | no pretraining, fully understandable, fast O(n×d) |
| Weaknesses | weak at semantic matching ("car" ≠ "automobile"), no understanding of context |
| L89 | assemble this retriever into a complete RAG pipeline |

Next lesson (L89) combines TF-IDF retrieval, vector similarity and LLM generation into a complete RAG pipeline that answers questions end to end.

---

→ **Next lesson** · [L89 · A complete RAG pipeline](L89_rag_pipeline.ipynb)

> Next lesson covers **a complete RAG pipeline**: document chunking + TF-IDF retrieval + prompt construction + generation.